# Baseline A — GRU multimodal baseline (JAAD)
GRU multimodal baseline *inspired by* conventional pedestrian-intention models (not a PCPA reproduction).
Runs: controls (majority, last-frame logistic regression) → ablations 1-3 (box → +ego speed → +velocity)
on JAAD_beh and JAAD_all, 3 seeds each → optional visual ablations 4-5 (pedestrian-crop and scene Swin features).

**Kaggle settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** ·
**Add Input → Datasets → `christopheronyiuke/jaad-video`** (346 official JAAD mp4 clips, mounted at
`/kaggle/input/jaad-video/JAAD_clips`). If the dataset is not attached, the notebook falls back to downloading the
official 3.1 GB zip from York University.

## Config

In [ ]:
REPO_URL = 'https://github.com/Vidit-01/pedestrian_intent_prediction.git'   # <-- set to your repo

WORK = '/kaggle/working'
REPO, JAAD, SAMPLES = f'{WORK}/repo', f'{WORK}/JAAD', f'{WORK}/samples'
CLIPS = '/kaggle/input/jaad-video/JAAD_clips'   # attached dataset christopheronyiuke/jaad-video
SUBSETS = ['beh', 'all']
SEEDS = [0, 1, 2]
KIN_ABLATIONS = {'box': ['box'], 'box_speed': ['box', 'speed'], 'box_speed_vel': ['box', 'speed', 'vel']}

RUN_VISUAL_ABLATIONS = False      # ablations 4-5: need clips + Swin feature extraction (~1 h on 2x T4 for JAAD_beh)
VISUAL_SUBSETS = ['beh']
FEATS = '/tmp/jaad_feats'         # feature cache (big; /kaggle/working is capped at 20 GB)
FEATS_FROM_DATASET = None         # or a path under /kaggle/input/... with features saved by an earlier run
NEED_CLIPS = RUN_VISUAL_ABLATIONS and not FEATS_FROM_DATASET

## 1. Code

In [ ]:
!git clone -q {REPO_URL} {REPO} 2>/dev/null || git -C {REPO} pull -q
%cd {REPO}
!git log -1 --format='code commit: %H'
!pip install -q -U timm
!nvidia-smi -L
!df -h /kaggle/working /tmp

## 2. Dataset

In [ ]:
# Official JAAD annotations + split_ids + jaad_data.py (GitHub, ~50 MB). Clips: symlinked from the attached Kaggle
# dataset, or downloaded from York if it is not attached.
import os
if not NEED_CLIPS:
    clips_arg = ''
elif os.path.isdir(CLIPS):
    clips_arg = f'--clips_from {CLIPS}'
else:
    print('Kaggle dataset not attached -> downloading the official clips zip (~3.1 GB)')
    clips_arg = '--clips'
!python download_jaad.py --root {JAAD} {clips_arg}

## Samples and split
PCPA benchmark protocol (also used by ACIT): official video split (177 train / 29 val / 117 test videos),
16 observed frames ending 30-60 frames (1-2 s) before the crossing event, windows with 80 % overlap (stride 3),
tracks shorter than 76 frames dropped. Label = whether the pedestrian crosses. JAAD_beh = behaviour-annotated
pedestrians; JAAD_all adds the non-behavioural ones (label 0).

In [ ]:
for sub in SUBSETS:
    !python build_jaad_samples.py --jaad_root {JAAD} --out {SAMPLES} --sample_type {sub}

import pandas as pd
for sub in SUBSETS:
    man = pd.read_csv(f'{SAMPLES}/jaad_{sub}_manifest.csv')
    print(f'JAAD_{sub}'); display(man.groupby('split').agg(videos=('vid', 'nunique'), peds=('pid', 'nunique'),
                                                           samples=('label', 'size'), positives=('label', 'sum')))

## 3. Training helper — both T4s busy
The GRU is tiny, so instead of DDP we run several independent runs at once, round-robin over the two GPUs
(`per_gpu` concurrent jobs each). Logs go to `logs/<run>.log`.

In [ ]:
import subprocess, time
os.makedirs('logs', exist_ok=True)

def run_parallel(jobs, n_gpus=2, per_gpu=2):
    """jobs: list of (run_name, [args for train.py]). Skips runs that already have metrics.json."""
    jobs = [j for j in jobs if not os.path.exists(f'runs/{j[0]}/metrics.json')]
    running, slot = [], 0
    while jobs or running:
        for n, p in [r for r in running if r[1].poll() is not None]:
            print(f'done {n} (exit {p.returncode})')
            running.remove((n, p))
        while jobs and len(running) < n_gpus * per_gpu:
            name, args = jobs.pop(0)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(slot % n_gpus)); slot += 1
            log = open(f'logs/{name}.log', 'w')
            running.append((name, subprocess.Popen(['python', 'train.py', *args, '--out', f'runs/{name}'],
                                                   stdout=log, stderr=subprocess.STDOUT, env=env)))
        time.sleep(5)

## 4. Controls: majority class and last-frame logistic regression

In [ ]:
for sub in SUBSETS:
    !python simple_baselines.py --samples {SAMPLES}/jaad_{sub}.pkl --out runs/simple_{sub}

## 5. Ablations 1-3: kinematic inputs

In [ ]:
jobs = [(f'gru_{sub}_{tag}_s{seed}',
         ['--model', 'gru', '--samples', f'{SAMPLES}/jaad_{sub}.pkl', '--kin_feats', *feats, '--seed', str(seed)])
        for sub in SUBSETS for tag, feats in KIN_ABLATIONS.items() for seed in SEEDS]
run_parallel(jobs)
!tail -n 1 logs/gru_*_s0.log

## 6. Ablations 4-5: + pedestrian-crop features, + scene features (optional)
Mean-pooled frozen Swin-V2-B features of the pedestrian crop (`local_rgb`) and the full frame (`global_rgb`),
each through its own GRU branch, fused with the kinematic branch. Set `RUN_VISUAL_ABLATIONS = True` in the config.

In [ ]:
if RUN_VISUAL_ABLATIONS:
    if FEATS_FROM_DATASET:
        FEATS = FEATS_FROM_DATASET
    else:
        src = f'{SAMPLES}/jaad_all.pkl' if 'all' in VISUAL_SUBSETS else f'{SAMPLES}/jaad_beh.pkl'
        cmd = (f'torchrun --nproc_per_node=2 extract_jaad_visual.py --jaad_root {JAAD} --samples {src} --out {FEATS} '
               '--modalities local_rgb global_rgb --vis 2')
        !{cmd}
    base = ['--model', 'gru', '--feats', FEATS, '--kin_feats', 'box', 'speed', 'vel']
    jobs = [(f'gru_{sub}_{tag}_s{seed}', base + ['--samples', f'{SAMPLES}/jaad_{sub}.pkl', '--visual', *vis, '--seed', str(seed)])
            for sub in VISUAL_SUBSETS for tag, vis in {'crop': ['local_rgb'], 'crop_scene': ['local_rgb', 'global_rgb']}.items()
            for seed in SEEDS]
    run_parallel(jobs, per_gpu=1)

## Results
`bench_*`: PCPA-benchmark / paper style at threshold 0.5, where **`bench_auc` is ROC-AUC of the rounded predictions**
(that's the number published tables report). `auc`: real ROC-AUC. `thr_*`: threshold chosen on validation.

In [ ]:
import glob
from evaluate import results_table
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
table = results_table(glob.glob('runs/gru_*') + glob.glob('runs/simple_*/*'))
table.to_csv('runs/baseline_a_results.csv', index=False)
display(table)

In [ ]:
!zip -qr /kaggle/working/results.zip runs logs -x "*.pt"
!ls -lh /kaggle/working/results.zip